## 02: Materialize Silver in Unity Catalog

Reads bronze_player_events (UC), applies the same build_silver used locally,
writes silver_player_events as a UC managed table. The point is not the numbers
(this is the 3-day Volume subset, not 50M): it is the lineage edge
bronze -> silver that UC captures because we read one UC table and saveAsTable
another. Serverless: spark already exists, no session bootstrap.

In [0]:
# %%
import os
from src.ingestion.config import load_source_config
from src.ingestion.silver import build_silver

CATALOG = "workspace"
SCHEMA = "telemetry"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_player_events"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_player_events"

# CWD is notebooks/databricks/, so config/ is two levels up. Derive the repo root
# from the notebook's own location instead of hardcoding the absolute path, so the
# notebook survives a move or a different user's Workspace. Same CWD lesson as local.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CONFIG_PATH = os.path.join(REPO_ROOT, "config", "sources", "player_events.yaml")

# Single source of truth: the same YAML the local pipeline uses. Rebuilding the
# config by hand would fork identity_key / ordering_key / dedup_window_hours away
# from local, and the dedup contract must be identical across environments.
cfg = load_source_config(CONFIG_PATH)
print("loaded config:", cfg.name, "| identity:", cfg.identity_key, "| order:", cfg.ordering_key)

In [0]:
# %%
# Read the UC table by NAME (spark.table), not by path. Reading by name is what
# makes UC record bronze as an upstream of whatever we write next. A path read
# would not attach to the lineage graph.
bronze = spark.table(BRONZE_TABLE)
print("bronze rows (3-day subset):", bronze.count())

In [0]:
# %%
# Same transformation as local. build_silver dedups within the window using the
# producer sequence, identical behavior to the 50M local run, just less data.
silver = build_silver(bronze, cfg)
print("silver rows after dedup:", silver.count())

In [0]:
# %%
# saveAsTable (managed UC table), NOT save(path). This is the write that registers
# the table in the catalog and closes the lineage edge bronze -> silver. Overwrite
# so the notebook is idempotent: re-running gives the same table, not a duplicate.
(
    silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")  # silver schema differs from bronze (dedup drops nothing but this is safe on re-run)
    .saveAsTable(SILVER_TABLE)
)
print("wrote", SILVER_TABLE)

In [0]:
# %%
# Confirm it landed in UC.
spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").show(truncate=False)